# LSTM Keyboard Language Model
Word-level next-token prediction · PyTorch

In [ ]:
# ── Датасет ───────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/nnytemirlan09a/keyboard/uzb_txt.txt"   # ← укажи свой файл
MODEL_PATH   = "lstm_model.pt" # куда сохранять модель

# ── Гиперпараметры ────────────────────────────────────────
EPOCHS     = 30
BATCH_SIZE = 64
SEQ_LEN    = 5    # контекстное окно (токенов)
EMBED_DIM  = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT    = 0.4
LR         = 2e-3
MIN_COUNT  = 5     # мин. частота слова для вхождения в словарь

## 📦 Импорты

In [ ]:
import re, math, json, time, pickle, random
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

## 🖥️ Устройство

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')

## 📄 Загрузка корпуса

In [ ]:
PAD, UNK, BOS, EOS = '<pad>', '<unk>', '<s>', '</s>'
SPECIAL = [PAD, UNK, BOS, EOS]
PAD_IDX = 0

def tokenize(text: str) -> List[str]:
    # апостроф оставляем (bo'lgan), дефис оставляем (top-toza),
    # но одиночное тире – не трогаем (оно отфильтруется само)
    return re.findall(r"[a-zA-Z\u0400-\u04FF\w]['\-\w]*", text.lower(), flags=re.UNICODE)

sentences = []
with open(DATASET_PATH, encoding='utf-8') as f:
    lines = f.readlines()

for line in lines[1:]:
    line = line.strip()
    if not line:
        continue
    if line.startswith('***'):   # пропускаем разделители глав
        continue
    if line.startswith('Mutolaa'):  # пропускаем строки-колонтитулы
        continue
    toks = tokenize(line)
    if toks:
        sentences.append(toks)

total_tokens = sum(len(s) for s in sentences)
print(f'Sentences : {len(sentences):,}')
print(f'Tokens    : {total_tokens:,}')
print(f'Example   : {sentences[0]}')

## 🔤 Словарь

In [ ]:
random.seed(42)
random.shuffle(sentences)
split = int(len(sentences) * 0.9)
train_sents = sentences[:split]
dev_sents   = sentences[split:] if sentences[split:] else sentences[:max(1, len(sentences)//10)]

counts = Counter(w for s in train_sents for w in s)
w2i = {w: i for i, w in enumerate(SPECIAL)}
for w, c in sorted(counts.items()):
    if c >= MIN_COUNT and w not in w2i:
        w2i[w] = len(w2i)
i2w = {i: w for w, i in w2i.items()}

print(f'Vocab size : {len(w2i):,}')
print(f'Train sents: {len(train_sents):,}  |  Dev sents: {len(dev_sents):,}')

## 🗃️ Dataset & DataLoader

In [ ]:
def encode(tokens):
    return [w2i.get(t, w2i[UNK]) for t in tokens]

class LMDataset(Dataset):
    def __init__(self, sents, seq_len):
        self.samples = []
        for sent in sents:
            ids = encode([BOS] + sent + [EOS])
            for i in range(1, len(ids)):
                ctx = ids[max(0, i - seq_len): i]
                if len(ctx) < seq_len:
                    ctx = [PAD_IDX] * (seq_len - len(ctx)) + ctx
                self.samples.append((ctx, ids[i]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        ctx, tgt = self.samples[idx]
        return torch.tensor(ctx, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

train_ds = LMDataset(train_sents, SEQ_LEN)
dev_ds   = LMDataset(dev_sents,   SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_ds):,}  |  Dev samples: {len(dev_ds):,}')

## 🧠 Модель

In [ ]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                             batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.drop  = nn.Dropout(dropout)
        self.fc    = nn.Linear(hidden_dim, vocab_size)

        nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        for name, p in self.lstm.named_parameters():
            if 'weight' in name: nn.init.orthogonal_(p)
            elif 'bias' in name: nn.init.zeros_(p)

    def forward(self, x):
        emb    = self.drop(self.embed(x))
        out, _ = self.lstm(emb)
        out    = self.drop(out[:, -1, :])
        return self.fc(out)

model = LSTMLanguageModel(
    vocab_size=len(w2i),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Параметров: {n_params:,}')
print(model)

## 🏋️ Обучение

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

best_val_loss = float('inf')

print(f"{'Epoch':>6}  {'Train loss':>11}  {'Val loss':>9}  {'PPL':>8}  {'Time':>7}")
print('─' * 52)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # train
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # eval
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in dev_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            val_loss += criterion(model(x), y).item()
    val_loss /= len(dev_loader)
    ppl = math.exp(min(val_loss, 20))

    scheduler.step(val_loss)
    elapsed = time.time() - t0

    marker = '  ✓ best' if val_loss < best_val_loss else ''
    print(f"{epoch:>6}  {train_loss:>11.4f}  {val_loss:>9.4f}  {ppl:>8.2f}  {elapsed:>6.1f}s{marker}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state': model.state_dict(),
            'w2i': w2i, 'i2w': i2w,
            'config': {
                'vocab_size': len(w2i),
                'embed_dim': EMBED_DIM, 'hidden_dim': HIDDEN_DIM,
                'num_layers': NUM_LAYERS, 'dropout': DROPOUT, 'seq_len': SEQ_LEN,
            }
        }, MODEL_PATH)

print(f'\nГотово. Лучший val loss: {best_val_loss:.4f}  PPL: {math.exp(min(best_val_loss,20)):.2f}')
print(f'Модель сохранена: {MODEL_PATH}')

## 🔮 Предсказание следующего слова

In [ ]:
# Загружаем лучший чекпоинт
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
cfg  = ckpt['config']
model_inf = LSTMLanguageModel(
    vocab_size=cfg['vocab_size'], embed_dim=cfg['embed_dim'],
    hidden_dim=cfg['hidden_dim'], num_layers=cfg['num_layers'], dropout=cfg['dropout'],
).to(DEVICE)
model_inf.load_state_dict(ckpt['model_state'])
model_inf.eval()
_w2i, _i2w = ckpt['w2i'], ckpt['i2w']

@torch.no_grad()
def predict(context: list, top_k=10, prefix='', temperature=1.0):
    """
    context  — список слов, например ['muz', 'ıssıq']
    prefix   — фильтр по началу слова, например 'q'
    """
    seq = [_w2i.get(BOS, 1)] + [_w2i.get(w, _w2i[UNK]) for w in context]
    seq = seq[-cfg['seq_len']:]
    if len(seq) < cfg['seq_len']:
        seq = [PAD_IDX] * (cfg['seq_len'] - len(seq)) + seq
    x = torch.tensor([seq], dtype=torch.long, device=DEVICE)
    probs = F.softmax(model_inf(x)[0] / temperature, dim=-1).cpu().tolist()
    skip = {PAD, UNK, BOS, EOS}
    results = [
        (_i2w[i], probs[i])
        for i in range(len(probs))
        if _i2w[i] not in skip
        and (not prefix or _i2w[i].startswith(prefix))
    ]
    results.sort(key=lambda x: -x[1])
    return results[:top_k]

print('Функция predict() готова.')

## 🧪 Тесты

In [ ]:
# Меняй контекст как хочешь
test_cases = [
    ([], ''),           # начало предложения
    (['muz'], ''),
    (['çay'], ''),
    (['qand', 'wa'], ''),
    (['muz'], 'q'),     # только слова на 'q'
    (['Men', 'seni'], ''),
    (['bug'], '')
]

for ctx, pfx in test_cases:
    preds = predict(ctx, top_k=5, prefix=pfx)
    label = ' '.join(ctx) if ctx else '(начало)'
    if pfx: label += f'  prefix={pfx!r}'
    print(f'\n{label}')
    for w, p in preds:
        bar = '█' * int(p * 50)
        print(f'  {w:20s} {p:.4f}  {bar}')

In [2]:
MODEL_NAME = "ai-forever/mGPT"
CORPUS_PATH = "data/merged2.txt"       


EPOCHS = 3
BATCH_SIZE = 4
LR = 2e-4
MAX_SEQ_LEN = 128
GRAD_ACCUM = 4                          
LORA_R = 16
SAMPLE_AFTER_TRAIN = True              

# ══════════════════════════════════════════════════════════════════════════

import math
import os
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel

# ── Device ───────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

# ── Load tokenizer & model ───────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type != "cpu" else torch.float32,
)
model = model.to(device)
print(f"Base params: {sum(p.numel() for p in model.parameters()):,}")

# ── LoRA ─────────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj", "c_fc"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Dataset ──────────────────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.max_len = max_len
        all_ids = []
        for text in tqdm(texts, desc="Tokenizing"):
            ids = tokenizer.encode(text, add_special_tokens=False)
            if ids:
                all_ids.extend(ids)
                all_ids.append(tokenizer.eos_token_id or 0)

        self.chunks = []
        stride = max_len // 2
        for i in range(0, len(all_ids) - max_len + 1, stride):
            chunk = all_ids[i : i + max_len]
            if len(chunk) == max_len:
                self.chunks.append(chunk)
        print(f"  → {len(self.chunks):,} chunks from {len(texts):,} texts")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        ids = torch.tensor(self.chunks[idx], dtype=torch.long)
        return ids, ids.clone()

def collate_fn(batch):
    inputs, targets = zip(*batch)
    return torch.stack(inputs), torch.stack(targets)

# ── Load corpus ──────────────────────────────────────────────────────────
print("\nLoading corpus...")
lines = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            lines.append(line)

n_val = max(100, int(len(lines) * 0.02))
train_lines = lines[:-n_val]
val_lines = lines[-n_val:]
print(f"  Train: {len(train_lines):,} | Val: {len(val_lines):,}")

train_dataset = TextDataset(train_lines, tokenizer, max_len=MAX_SEQ_LEN)
val_dataset = TextDataset(val_lines, tokenizer, max_len=MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# ── Optimizer & scheduler ────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=total_steps,
    pct_start=0.05, anneal_strategy="cos",
)

print(f"\nTotal steps: {total_steps:,}")

# ── Training ─────────────────────────────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'─'*50}")
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"{'─'*50}")

    model.train()
    total_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc="  Train")
    for batch_idx, (input_ids, labels) in enumerate(pbar):
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        loss = model(input_ids=input_ids, labels=labels).loss / GRAD_ACCUM
        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM
        n_batches += 1

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        avg = total_loss / n_batches
        pbar.set_postfix(loss=f"{avg:.4f}", ppl=f"{math.exp(min(avg, 20)):.0f}")

    # Validation
    model.eval()
    val_loss = 0.0
    val_n = 0
    with torch.no_grad():
        for input_ids, labels in tqdm(val_loader, desc="  Val  "):
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            val_loss += model(input_ids=input_ids, labels=labels).loss.item()
            val_n += 1
    val_loss /= val_n
    val_ppl = math.exp(min(val_loss, 20))

    print(f"  val_loss={val_loss:.4f} | val_ppl={val_ppl:.1f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  ✓ Saved best (ppl={val_ppl:.1f})")

print(f"\n{'='*50}")
print(f"Done! Best val PPL: {math.exp(min(best_val_loss, 20)):.1f}")
print(f"{'='*50}")

# ── Generation ───────────────────────────────────────────────────────────
if SAMPLE_AFTER_TRAIN:
    # Load best adapter
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if device.type != "cpu" else torch.float32,
    )
    model = PeftModel.from_pretrained(base, OUTPUT_DIR).to(device)
    model.eval()

    print(f"\nSamples:")
    for prompt in ["Men seni", "Bir kuni", "Bu dunyoda", "Qizil dengizning"]:
        ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                ids, max_new_tokens=30, temperature=0.8,
                top_p=0.9, top_k=50, do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        print(f"  {prompt} → {tokenizer.decode(out[0], skip_special_tokens=True)}")


Device: cuda
Loading tokenizer...


config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Val  : 100%|██████████| 39/39 [00:00<00:00, 48.84it/s]


  val_loss=3.5781 | val_ppl=35.8
  ✓ Saved best (ppl=35.8)

Done! Best val PPL: 35.8


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/mGPT
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Samples:
  Men seni → Men seni ko'rgan odamlar ham sening yoshligingdan boshqa bir narsani bilmaydilar, endi
  Bir kuni → Bir kuni yonimizda yurgan yigit o'ziga keldi.
  Bu dunyoda → Bu dunyoda menga sizlar kerak.
  Qizil dengizning → Qizil dengizning eng baland qirg'og'ida joylashgan bu shaharning o'rta qismida ikki guruh o


In [10]:
base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if device.type != "cpu" else torch.float32,
    )
model = PeftModel.from_pretrained(base, OUTPUT_DIR).to(device)
model.eval()

print(f"\nSamples:")
for prompt in ["Men seni", "Bir kuni", "Bu dunyoda", "Qizil dengizning", 'Türkis']:
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=50, temperature=0.8,
            top_p=0.9, top_k=50, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    print(f"  {prompt} → {tokenizer.decode(out[0], skip_special_tokens=True)}")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/mGPT
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Samples:
  Men seni → Men seni qiyomatda ko'rmasam edi.
  Bir kuni → Bir kuni uning o'ng boshiga qo'l solib, bir yoniga qo'yib o'tirgan qo'ltiqlari bilan ko'zlarini artdi.
  Bu dunyoda → Bu dunyoda sizga o'xshab ko'p yigitlar ham o'ldirilgan, men sizga o'xshab ko'p qizlar ham qonxo'r bo'lgan, men
  Qizil dengizning → Qizil dengizning sariq-qizil rangi qon tomirlarini o'rab olib, daryodan oqib o'tadigan asosiy yo'lni kesib o'tadi.
  Türkis → Türkis, iskash, balo-qazodan yiroq, bu dunyo shu yerda qoldi!


In [19]:
"""
Two generation modes for mGPT + LoRA:
  1. predict_next_word(context)  → целое следующее слово
  2. complete_token(prefix)      → дополнить начатый токен (sa → salem)

Before running:
    !pip install transformers peft accelerate torch
"""

# ══════════════════════════════════════════════════════════════════════════
#  НАСТРОЙКИ
# ══════════════════════════════════════════════════════════════════════════

MODEL_NAME = "ai-forever/mGPT"
ADAPTER_PATH = "mgpt-lora"
MAX_NEW_TOKENS = 50
TEMPERATURE = 0.8
TOP_P = 0.9

# ══════════════════════════════════════════════════════════════════════════

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ── Device ───────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# ── Load model ───────────────────────────────────────────────────────────
print(f"Loading model on {device}...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type != "cpu" else torch.float32,
)
model = PeftModel.from_pretrained(base, ADAPTER_PATH).to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Ready!\n")




def predict_next_word(context: str, top_k: int = 5, temperature: float = 0.8) -> list[tuple[str, float]]:
    """
    Предсказывает следующее целое слово после контекста.
    Возвращает top_k вариантов с вероятностями.

    Пример: "Men seni" → [("yaxshi", 0.32), ("bilaman", 0.18), ...]
    """
    # Кодируем контекст
    input_ids = tokenizer.encode(context, return_tensors="pt").to(device)
    context_len = input_ids.shape[1]

    # Генерируем несколько токенов
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=temperature,
            top_p=TOP_P,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=top_k,
        )

    # Декодируем и извлекаем первое слово после контекста
    words = []
    for out in outputs:
        # Декодируем только новые токены
        new_tokens = out[context_len:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True)

        # Берём первое слово (до пробела или конца)
        first_word = text.split()[0] if text.split() else ""
        if first_word:
            # Чистим от пунктуации
            first_word = re.sub(r'^[^\w]+|[^\w]+$', '', first_word)
            if first_word:
                words.append(first_word)

    # Считаем частоту → вероятность
    from collections import Counter
    freq = Counter(words)
    total = sum(freq.values()) or 1
    results = [(w, round(c / total, 3)) for w, c in freq.most_common(top_k)]
    return results


# ══════════════════════════════════════════════════════════════════════════
#  2. COMPLETE TOKEN (prefix → full word)
#
#  "sa" → генерируем продолжение пока не пробел/EOS
#  Возвращаем дополненное слово
# ══════════════════════════════════════════════════════════════════════════

def complete_token(prefix: str, top_k: int = 5, temperature: float = 0.9) -> list[tuple[str, float]]:
    """
    Дополняет начатый токен до целого слова.

    Пример: "sa" → [("salem", 0.35), ("sag'ol", 0.22), ("salom", 0.18), ...]
    """
    prefix = prefix.strip().lower()
    if not prefix:
        return []

    # Кодируем префикс
    input_ids = tokenizer.encode(prefix, return_tensors="pt").to(device)
    prefix_len = input_ids.shape[1]

    # Генерируем продолжения
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=20,  # достаточно для одного слова
            temperature=temperature,
            top_p=TOP_P,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=top_k * 3,  # больше вариантов для разнообразия
        )

    completions = []
    for out in outputs:
        # Декодируем только новые токены
        new_tokens = out[prefix_len:]
        suffix = tokenizer.decode(new_tokens, skip_special_tokens=True)

        # Берём продолжение до первого пробела
        suffix_word = suffix.split()[0] if suffix.split() else ""

        # Склеиваем с префиксом
        full_word = prefix + suffix_word

        # Убираем дубликаты префикса (если модель повторила начало)
        if full_word.startswith(prefix + prefix):
            full_word = full_word[len(prefix):]

        if full_word and full_word != prefix:
            completions.append(full_word)

    # Считаем частоту → вероятность
    from collections import Counter
    freq = Counter(completions)
    total = sum(freq.values()) or 1
    results = [(w, round(c / total, 3)) for w, c in freq.most_common(top_k)]
    return results


# ══════════════════════════════════════════════════════════════════════════
#  DEMO
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("1. NEXT WORD PREDICTION")
    print("=" * 60)

    for ctx in ["Men seni", "Bir kuni", "Bu dunyoda", "Qizil dengizning"]:
        words = predict_next_word(ctx, top_k=5)
        print(f"\n  {ctx} →")
        for w, p in words:
            print(f"    {w:>15}  {p:.1%}")

    print("\n" + "=" * 60)
    print("2. TOKEN COMPLETION")
    print("=" * 60)

    for prefix in ["sa", "ki", "bo", "qo", "men", "bir", 'bu', 'bug', 'pay', 'Qamal']:
        completions = complete_token(prefix, top_k=5)
        print(f"\n  {prefix} →")
        for w, p in completions:
            print(f"    {w:>15}  {p:.1%}")

Loading model on cuda...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/mGPT
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready!

1. NEXT WORD PREDICTION

  Men seni →
              menga  20.0%
          o'ldirdim  20.0%
       ko'rmayapman  20.0%
              kuyav  20.0%
                ham  20.0%

  Bir kuni →
               soat  20.0%
      yuz-ko'zlarim  20.0%
                o'z  20.0%
              Tohir  20.0%
           Hasanali  20.0%

  Bu dunyoda →
      tushunilmagan  20.0%
             qizlar  20.0%
             qancha  20.0%
             o'ziga  20.0%
               ko'p  20.0%

  Qizil dengizning →
              qaysi  20.0%
             pastki  20.0%
       qirg'og'idan  20.0%
              quyuq  20.0%
           sohilida  20.0%

2. TOKEN COMPLETION

  sa →
                sa,  26.7%
    sakengaytirmoqchi  6.7%
            sayo'q,  6.7%
            saxayol  6.7%
            samunda  6.7%

  ki →
                ki,  20.0%
         kiyiqilib,  13.3%
         kiborlig'i  6.7%
       kiso'zlarini  6.7%
        kiboshqacha  6.7%

  bo →
             bo'lsa  13.3%
            bo'ldi.  13.3

In [ ]:
input(

)